In [51]:
import torch
import torch.nn as nn

class LSTM(nn.Module):
    def __init__(self, input_size=12, num_target_features=2, hidden_size=5, num_layers=3, dropout=0.3, bidirectional=False):
        super(LSTM, self).__init__()
        self.num_target_features = num_target_features
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.lstm = nn.LSTM(
            input_size, 
            hidden_size, 
            num_layers, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )

        direction_factor = 2 if bidirectional else 1
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * direction_factor, 128),
            nn.ReLU(),
            nn.Linear(128, num_target_features)
        )

    def forward(self, x):
        h0 = torch.zeros(self.num_layers * (2 if self.bidirectional else 1), x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers * (2 if self.bidirectional else 1), x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        print(out.shape)
        out = out[:, -1, :] 
        out = self.fc(out)
        out = out.view(x.size(0), self.num_target_features)
        return out

In [54]:
model = LSTM(input_size=5, num_target_features=2, hidden_size=7, num_layers=3, dropout=0.3, bidirectional=False)

In [55]:
X = torch.randn(5,5,5)
# print(X.shape)
y_pred = model(X)
print(y_pred)

torch.Size([5, 5, 7])
tensor([[ 0.0479, -0.0443],
        [ 0.0551, -0.0495],
        [ 0.0474, -0.0452],
        [ 0.0574, -0.0494],
        [ 0.0403, -0.0455]], grad_fn=<ViewBackward0>)
